In [1]:
# ============================================================
# Notebook    : 00_data_collection.ipynb (rewritten)
# Description : Collect three datasets for UBI underwriting research
#               - freMTPL2 (HuggingFace)
#               - fremotor2freq9907b (CASdatasets via R script)
#               - car-insurance-data (Kaggle)
#
#               CHANGES FROM PRIOR VERSION:
#               - R subprocess output streams live (was: silent
#                 until process exits, indistinguishable from hang)
#               - R subprocess has a hard timeout (was: none,
#                 could wait forever)
#               - CASdatasets install is forced, no
#                 requireNamespace skip (was: could silently skip
#                 install and fail later on a stale lib path)
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install datasets kaggle pandas


# ============================================================
# 1. Common imports
# ============================================================
import os
import subprocess
import glob
import sys
import pandas as pd

os.makedirs("data", exist_ok=True)


# ============================================================
# 2. Dataset 1 — freMTPL2 (HuggingFace: mabilton/fremtpl2)
# ============================================================
from datasets import load_dataset

ds_freq = load_dataset("mabilton/fremtpl2", "freMTPL2freq")
ds_sev  = load_dataset("mabilton/fremtpl2", "freMTPL2sev")

df_freq = ds_freq["train"].to_pandas()
df_sev  = ds_sev["train"].to_pandas()

print("freMTPL2freq shape:", df_freq.shape)
print("freMTPL2sev  shape:", df_sev.shape)
print(df_freq.head(3))

df_freq.to_csv("data/freMTPL2freq.csv", index=False)
df_sev.to_csv("data/freMTPL2sev.csv",  index=False)
print("freMTPL2 saved.")


# ============================================================
# 3. Dataset 2 — fremotor2freq9907b (CASdatasets via R script)
#
#    NOTE: writes get_casdatasets.R fresh every run — forced
#    reinstall of CASdatasets, no requireNamespace skip, so a
#    changed local environment (new R version, moved lib path,
#    etc.) can't silently reuse a stale/missing install.
# ============================================================
r_script_content = '''
Sys.setenv(R_DEFAULT_INTERNET_TIMEOUT = "600")
options(timeout = 600)
options(download.file.method = "libcurl")
user_lib <- file.path(Sys.getenv("USERPROFILE"), "R", "win-library", "4.6")
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))

cat("=== .libPaths() ===\\n")
print(.libPaths())

cat("\\n=== Installing xts ===\\n")
install.packages("xts", repos = "https://cloud.r-project.org", lib = user_lib)

cat("\\n=== Installing CASdatasets (forced) ===\\n")
install.packages(
  "CASdatasets",
  repos = "https://dutangc.perso.math.cnrs.fr/RRepository/pub/",
  type  = "source",
  lib   = user_lib
)

cat("\\n=== Verifying CASdatasets install ===\\n")
if (!requireNamespace("CASdatasets", lib.loc = user_lib, quietly = TRUE)) {
  stop("CASdatasets install FAILED.")
}
cat("CASdatasets install OK.\\n")

cat("\\n=== Loading fremotor2freq9907b ===\\n")
library(CASdatasets, lib.loc = user_lib)
data(fremotor2freq9907b)
write.csv(fremotor2freq9907b, "data/fremotor2freq9907b.csv", row.names = FALSE)
cat("fremotor2freq9907b saved.\\n")
'''
with open("get_casdatasets.R", "w", encoding="utf-8") as f:
    f.write(r_script_content)
print("get_casdatasets.R written (forced reinstall).")

rscript_candidates = glob.glob(r"C:\Program Files\R\R-*\bin\Rscript.exe")

if not rscript_candidates:
    print("WARNING: Rscript.exe not found under C:\\Program Files\\R\\")
    print("Install R from https://cran.r-project.org/bin/windows/base/")
    RSCRIPT_PATH = None
else:
    RSCRIPT_PATH = rscript_candidates[0]
    print("Found Rscript at:", RSCRIPT_PATH)

df_motor = None
R_TIMEOUT_SEC = 900  # 15 min hard ceiling — CASdatasets source build
                      # normally finishes in 1-3 min; if it's still
                      # running past this, something is actually stuck
                      # (network stall, compiler prompt, etc.), not
                      # just "slow"

if RSCRIPT_PATH and os.path.exists("get_casdatasets.R"):
    print(f"\nRunning get_casdatasets.R (timeout={R_TIMEOUT_SEC}s, "
          f"output streams live below)...\n" + "="*60)

    process = subprocess.Popen(
        [RSCRIPT_PATH, "get_casdatasets.R"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,   # merge stderr into stdout so
                                    # the interleaved order matches
                                    # what R actually printed
        text=True,
        encoding="utf-8",
        bufsize=1,                  # line-buffered
        cwd=os.getcwd()
    )

    output_lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)   # stream live, no buffering
            output_lines.append(line)
        returncode = process.wait(timeout=R_TIMEOUT_SEC)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
        print("\n" + "="*60)
        print(f"!! TIMEOUT after {R_TIMEOUT_SEC}s — process killed.")
        print("!! This means it was genuinely stuck (not just slow).")
        print("!! Common causes: network stall reaching the CRAN/CASdatasets")
        print("!! mirror, or a source compile waiting on a missing toolchain.")
        returncode = -1

    print("="*60)

    if returncode != 0:
        print(f"\n!! R script exited with code {returncode}.")
        print("!! Scroll up to the last printed section header (=== ... ===)")
        print("!! to see exactly which step failed.")
    else:
        df_motor = pd.read_csv("data/fremotor2freq9907b.csv")
        print("\nfremotor2freq9907b shape:", df_motor.shape)
        print(df_motor.head(3))
else:
    if not os.path.exists("get_casdatasets.R"):
        print("WARNING: get_casdatasets.R not found in current folder.")


# ============================================================
# 4. Dataset 3 — car-insurance-data (Kaggle: sagnik1511)
#
#    NOTE: requires kaggle.json at ~/.kaggle/kaggle.json
# ============================================================
import kaggle

kaggle.api.authenticate()
kaggle.api.dataset_download_files(
    "sagnik1511/car-insurance-data",
    path  = "data/car_insurance",
    unzip = True
)

df_car = pd.read_csv("data/car_insurance/Car_Insurance_Claim.csv")

print("car-insurance-data shape:", df_car.shape)
print(df_car.head(3))

df_car.to_csv("data/car_insurance.csv", index=False)
print("car-insurance-data saved.")


# ============================================================
# 5. Summary check
# ============================================================
datasets = {
    "freMTPL2freq"       : df_freq,
    "freMTPL2sev"        : df_sev,
    "car_insurance"      : df_car,
}
if df_motor is not None:
    datasets["fremotor2freq9907b"] = df_motor

print("\n===== Dataset Summary =====")
for name, df in datasets.items():
    print(f"{name:25s} | rows: {df.shape[0]:>7,} | cols: {df.shape[1]:>3}")
print("===========================")

if df_motor is None:
    print("\n⚠ fremotor2freq9907b was NOT loaded.")
    print("  Scroll up to the streamed R output to find the exact")
    print("  failure point (network / compile / verification step).")

C:\Users\User\anaconda3\envs\diceml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


freMTPL2freq shape: (678013, 12)
freMTPL2sev  shape: (26639, 2)
   IDpol  ClaimNb  Exposure  VehPower  VehAge  DrivAge  BonusMalus VehBrand  \
0      1        1      0.10         5       0       55          50      B12   
1      3        1      0.77         5       0       55          50      B12   
2      5        1      0.75         6       2       52          50      B12   

    VehGas Area  Density       Region  
0  Regular    D     1217  Rhone-Alpes  
1  Regular    D     1217  Rhone-Alpes  
2   Diesel    B       54     Picardie  
freMTPL2 saved.
get_casdatasets.R written (forced reinstall).
Found Rscript at: C:\Program Files\R\R-4.6.1\bin\Rscript.exe

Running get_casdatasets.R (timeout=900s, output streams live below)...
=== .libPaths() ===
[1] "C:/Users/User/R/win-library/4.6"              
[2] "C:/Users/User/AppData/Local/R/win-library/4.6"
[3] "C:/Program Files/R/R-4.6.1/library"           

=== Installing xts ===
trying URL 'https://cloud.r-project.org/bin/windows/contrib/4.6/